# Toxic Comment Classification — v8 (Hierarchical `severe_toxic` Head)

**Goal.** v7 capped `severe_toxic` test PR AUC at **0.344**, far below the head classes (0.68–0.80). The hypothesis: `severe_toxic` is structurally nested inside `toxic` (nearly every severe_toxic sample is also toxic; prevalence is ~1%). Treating it as an independent binary label forces the model to learn the rare boundary from all 159K samples, most of which provide weak signal.

**Approach.** Factor the label as `P(severe) = P(severe | toxic=1) · P(toxic)`:

1. Take v7's frozen trunk representation (the 384-dim hidden just before the final classifier).
2. Train a tiny `Linear(384, 1)` head on **only toxic-positive rows** to predict `severe_toxic`. This head sees a dense signal — the severe rate within toxic-positive rows is ~10%, not 1%.
3. At inference, combine: `P(severe)_final = P(severe | toxic=1) · P(toxic)_v7`.

**Why frozen backbone first.** v7's trunk already encodes "toxicity direction"; the conditional distinction is a near-linear split on top of it. If a logistic regression on trunk features moves PR AUC, that's the proof-of-concept. If it doesn't, unfreezing won't save it — the trunk doesn't have the information.

**Baseline to beat.** v7 severe_toxic test PR AUC = **0.344**, ROC AUC = 0.990 (ranking is near-perfect; PR AUC is starved by tiny positives).

**Story for portfolio.** Hierarchical decomposition of a nested multi-label problem — a named technique (cascade classifiers, coarse-to-fine) with a clear motivation grounded in the label structure, not just "try a new model."


In [1]:
# Imports
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score,
    recall_score, f1_score, precision_recall_curve,
)
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_float32_matmul_precision('high')
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


c:\Users\berke\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda (NVIDIA GeForce GTX 1070)


In [2]:
# Config — mirror v7 on architectural keys so the checkpoint loads cleanly.
# Only the conditional head is new; everything else is reused from v7.
CONFIG = {
    # Must match v7's saved config — these drive model instantiation
    'MAX_LENGTH': 192,
    'USE_EMBEDDINGS_ONLY': False,
    'FREEZE_LAYERS': 2,
    'HEAD_HIDDEN_SIZE': 384,
    'BATCH_SIZE': 64,
    # v8-specific — the conditional head training
    'COND_HEAD_EPOCHS': 20,
    'COND_HEAD_LR': 1e-3,
    'COND_HEAD_WEIGHT_DECAY': 1e-4,
    # Artifacts from v7 we consume
    'V7_CHECKPOINT': 'best_model_v7.pt',
    'V7_TEST_NPZ': 'test_inference_v7.npz',
    'V7_VAL_NPZ':  'val_inference_v7.npz',
}
labels_list = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


In [3]:
# Load raw data — we need the text + toxic/severe labels to build the toxic-positive
# subset for training the conditional head.
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
test_labels_df = pd.read_csv('test_labels.csv')

# Match v7's stratified split exactly — same seed, same proportions
from sklearn.model_selection import train_test_split
train_split, val_split = train_test_split(
    train_df, test_size=0.1, random_state=SEED,
    stratify=train_df['toxic'],
)
train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)
print(f"Train: {len(train_split):,}  Val: {len(val_split):,}")

# Valid test (exclude Kaggle's -1 rows)
valid_mask = (test_labels_df[labels_list] != -1).all(axis=1)
valid_test_df = test_df[valid_mask].reset_index(drop=True)
valid_test_labels = test_labels_df[valid_mask][labels_list].reset_index(drop=True).values
print(f"Valid test: {len(valid_test_df):,}")


Train: 143,613  Val: 15,958
Valid test: 63,978


In [4]:
# Severe-within-toxic statistics — this is what makes the hierarchical head tractable
for name, df in [('train', train_split), ('val', val_split)]:
    n = len(df)
    n_toxic = int(df['toxic'].sum())
    n_severe = int(df['severe_toxic'].sum())
    n_severe_and_toxic = int(((df['toxic'] == 1) & (df['severe_toxic'] == 1)).sum())
    n_severe_not_toxic = int(((df['toxic'] == 0) & (df['severe_toxic'] == 1)).sum())
    print(f"[{name}] n={n:,}")
    print(f"  toxic=1:                   {n_toxic:,} ({n_toxic/n:.2%})")
    print(f"  severe_toxic=1:            {n_severe:,} ({n_severe/n:.2%})")
    print(f"  severe AND toxic:          {n_severe_and_toxic:,}")
    print(f"  severe AND NOT toxic:      {n_severe_not_toxic:,}   <- 'leakage' from the nested assumption")
    if n_toxic > 0:
        print(f"  P(severe | toxic=1):       {n_severe_and_toxic/n_toxic:.2%}")
    print()


[train] n=143,613
  toxic=1:                   13,765 (9.58%)
  severe_toxic=1:            1,446 (1.01%)
  severe AND toxic:          1,446
  severe AND NOT toxic:      0   <- 'leakage' from the nested assumption
  P(severe | toxic=1):       10.50%

[val] n=15,958
  toxic=1:                   1,529 (9.58%)
  severe_toxic=1:            149 (0.93%)
  severe AND toxic:          149
  severe AND NOT toxic:      0   <- 'leakage' from the nested assumption
  P(severe | toxic=1):       9.74%



## Step 1 — Load v7 backbone frozen

We re-declare `MultiLabelDistilBert` verbatim from v7 so the checkpoint loads. All parameters are frozen and the model is put in eval mode — we only read from it.


In [5]:
# Model class — verbatim copy from v7 so the state_dict loads with strict=True
class MultiLabelDistilBert(nn.Module):
    def __init__(self, num_labels=6, dropout=0.1, use_embeddings_only=False,
                 freeze_layers=0, head_hidden_size=384):
        super().__init__()
        self.use_embeddings_only = use_embeddings_only
        self.freeze_layers = freeze_layers
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        bert_hidden_size = self.bert.config.hidden_size
        self.trunk = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(bert_hidden_size, head_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(head_hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        if self.use_embeddings_only:
            pooled = self.bert.embeddings(input_ids)[:, 0, :]
        else:
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            pooled = out.last_hidden_state[:, 0, :]
        return self.classifier(self.trunk(pooled))

    def trunk_features(self, input_ids, attention_mask):
        if self.use_embeddings_only:
            pooled = self.bert.embeddings(input_ids)[:, 0, :]
        else:
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            pooled = out.last_hidden_state[:, 0, :]
        return self.trunk(pooled)


In [6]:
# Instantiate and load v7 weights. Config guard: if the saved config's arch keys
# disagree with CONFIG here, fail loudly rather than silently loading mismatched weights.
v7 = MultiLabelDistilBert(
    num_labels=6,
    use_embeddings_only=CONFIG['USE_EMBEDDINGS_ONLY'],
    freeze_layers=CONFIG['FREEZE_LAYERS'],
    head_hidden_size=CONFIG['HEAD_HIDDEN_SIZE'],
).to(device)

ckpt = torch.load(CONFIG['V7_CHECKPOINT'], map_location=device, weights_only=False)
saved_cfg = ckpt.get('config', {})
arch_keys = ('MAX_LENGTH', 'USE_EMBEDDINGS_ONLY', 'FREEZE_LAYERS', 'HEAD_HIDDEN_SIZE')
mismatches = [k for k in arch_keys if saved_cfg.get(k) != CONFIG.get(k)]
if mismatches:
    raise RuntimeError(f"v7 checkpoint config mismatches: {mismatches}\n"
                       f"  saved: {[(k, saved_cfg.get(k)) for k in mismatches]}\n"
                       f"  current: {[(k, CONFIG.get(k)) for k in mismatches]}")

v7.load_state_dict(ckpt['model_state_dict'])
v7.eval()
for p in v7.parameters():
    p.requires_grad = False
print(f"✓ Loaded v7 from {CONFIG['V7_CHECKPOINT']}")
print(f"  Saved val PR AUC: {ckpt.get('val_pr_auc', 'n/a'):.4f}")
print(f"  Total params (frozen): {sum(p.numel() for p in v7.parameters()):,}")

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


✓ Loaded v7 from best_model_v7.pt
  Saved val PR AUC: 0.8765
  Total params (frozen): 66,660,486


## Step 2 — Extract trunk features for all splits

We forward every sample through the frozen v7 backbone + trunk once, saving the 384-dim trunk output to disk. After this, the conditional head trains as a plain logistic regression on NumPy arrays — no GPU, seconds not minutes.

Resume guard: each split caches to `trunk_features_{split}_v8.npz`. If the file exists, we skip the forward pass.


In [7]:
# Tokenization helper — variable-length, no padding (batched collate pads per batch)
def tokenize_text(text):
    enc = tokenizer(
        text, truncation=True, max_length=CONFIG['MAX_LENGTH'],
        padding=False, return_tensors=None,
    )
    return enc['input_ids'], enc['attention_mask']


class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        ids, mask = tokenize_text(str(self.texts[idx]))
        return {'input_ids': torch.tensor(ids, dtype=torch.long),
                'attention_mask': torch.tensor(mask, dtype=torch.long)}


def collate_varlen(batch):
    ids = pad_sequence([b['input_ids'] for b in batch], batch_first=True, padding_value=0)
    msk = pad_sequence([b['attention_mask'] for b in batch], batch_first=True, padding_value=0)
    return {'input_ids': ids, 'attention_mask': msk}


@torch.no_grad()
def extract_trunk_features(texts, desc='Extracting trunk features'):
    ds = TextDataset(list(texts))
    dl = DataLoader(ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=False,
                    num_workers=0, collate_fn=collate_varlen, pin_memory=True)
    feats = []
    for batch in tqdm(dl, desc=desc, leave=True):
        ids = batch['input_ids'].to(device, non_blocking=True)
        msk = batch['attention_mask'].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            h = v7.trunk_features(ids, msk)
        feats.append(h.float().cpu().numpy())
    return np.concatenate(feats, axis=0)


In [8]:
# Train split trunk features (all rows — we slice toxic-positive later)
_f = 'trunk_features_train_v8.npz'
if os.path.exists(_f):
    _d = np.load(_f)
    train_trunk = _d['trunk']; train_labels_arr = _d['labels']
    print(f"✓ Loaded {_f}: trunk={train_trunk.shape}, labels={train_labels_arr.shape}")
else:
    train_trunk = extract_trunk_features(train_split['comment_text'].values, desc='Train trunk')
    train_labels_arr = train_split[labels_list].values.astype(np.float32)
    np.savez(_f, trunk=train_trunk, labels=train_labels_arr)
    print(f"✓ Saved {_f}: trunk={train_trunk.shape}")


✓ Loaded trunk_features_train_v8.npz: trunk=(143613, 384), labels=(143613, 6)


In [9]:
# Val split trunk features (full val — we evaluate combined P(severe) on all val rows)
_f = 'trunk_features_val_v8.npz'
if os.path.exists(_f):
    _d = np.load(_f)
    val_trunk = _d['trunk']; val_labels_arr = _d['labels']
    print(f"✓ Loaded {_f}: trunk={val_trunk.shape}, labels={val_labels_arr.shape}")
else:
    val_trunk = extract_trunk_features(val_split['comment_text'].values, desc='Val trunk')
    val_labels_arr = val_split[labels_list].values.astype(np.float32)
    np.savez(_f, trunk=val_trunk, labels=val_labels_arr)
    print(f"✓ Saved {_f}: trunk={val_trunk.shape}")


✓ Loaded trunk_features_val_v8.npz: trunk=(15958, 384), labels=(15958, 6)


In [10]:
# Test split trunk features (all valid test rows)
_f = 'trunk_features_test_v8.npz'
if os.path.exists(_f):
    _d = np.load(_f)
    test_trunk = _d['trunk']
    print(f"✓ Loaded {_f}: trunk={test_trunk.shape}")
else:
    test_trunk = extract_trunk_features(valid_test_df['comment_text'].values, desc='Test trunk')
    np.savez(_f, trunk=test_trunk)
    print(f"✓ Saved {_f}: trunk={test_trunk.shape}")


✓ Loaded trunk_features_test_v8.npz: trunk=(63978, 384)


## Step 3 — Train the conditional head on toxic-positive rows

Filter train set to `toxic == 1`. Target is `severe_toxic`. Model is a single `Linear(384, 1)` with `BCEWithLogitsLoss`. Uses Adam + weight decay, no scheduler — this is overkill anyway for a tiny linear head.


In [11]:
# Build the toxic-positive training subset — this is where the conditional head sees
# a much denser positive rate (~10% severe vs 1% overall).
toxic_mask_train = (train_split['toxic'].values == 1)
X_cond_train = train_trunk[toxic_mask_train]
y_cond_train = train_split.loc[toxic_mask_train, 'severe_toxic'].values.astype(np.float32)
print(f"Conditional training set: {len(X_cond_train):,} rows")
print(f"  severe=1 within this subset: {int(y_cond_train.sum()):,} ({y_cond_train.mean():.2%})")

# Val: keep all rows for evaluating combined P(severe). For the head's own validation
# (conditional loss/PR), use the toxic-positive val subset.
toxic_mask_val = (val_split['toxic'].values == 1)
X_cond_val = val_trunk[toxic_mask_val]
y_cond_val = val_split.loc[toxic_mask_val, 'severe_toxic'].values.astype(np.float32)
print(f"Conditional val set:      {len(X_cond_val):,} rows  "
      f"(severe within: {int(y_cond_val.sum())})")


Conditional training set: 13,765 rows
  severe=1 within this subset: 1,446 (10.50%)
Conditional val set:      1,529 rows  (severe within: 149)


In [12]:
# Tiny conditional head: Linear(H, 1). Trained as a logistic regression.
torch.manual_seed(SEED); np.random.seed(SEED)

class ConditionalHead(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)
    def forward(self, x):
        return self.fc(x).squeeze(-1)


head = ConditionalHead(CONFIG['HEAD_HIDDEN_SIZE']).to(device)
opt = torch.optim.Adam(head.parameters(), lr=CONFIG['COND_HEAD_LR'],
                      weight_decay=CONFIG['COND_HEAD_WEIGHT_DECAY'])
bce = nn.BCEWithLogitsLoss()

# Batch the toxic-positive subset — small enough to fit comfortably in memory
X_tr = torch.tensor(X_cond_train, dtype=torch.float32, device=device)
y_tr = torch.tensor(y_cond_train, dtype=torch.float32, device=device)
X_va = torch.tensor(X_cond_val, dtype=torch.float32, device=device)
y_va = torch.tensor(y_cond_val, dtype=torch.float32, device=device)

BATCH = 256
history = []
best_val_pr = -1.0
best_state = None

for epoch in range(1, CONFIG['COND_HEAD_EPOCHS'] + 1):
    head.train()
    perm = torch.randperm(len(X_tr), device=device)
    total_loss = 0.0
    for i in range(0, len(X_tr), BATCH):
        idx = perm[i:i+BATCH]
        logits = head(X_tr[idx])
        loss = bce(logits, y_tr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * len(idx)
    train_loss = total_loss / len(X_tr)

    head.eval()
    with torch.no_grad():
        val_logits = head(X_va)
        val_loss = bce(val_logits, y_va).item()
        val_probs = torch.sigmoid(val_logits).cpu().numpy()
    val_pr = average_precision_score(y_cond_val, val_probs)
    val_roc = roc_auc_score(y_cond_val, val_probs)
    history.append({'epoch': epoch, 'train_loss': train_loss,
                    'val_loss': val_loss, 'val_pr_auc': val_pr, 'val_roc_auc': val_roc})
    if val_pr > best_val_pr:
        best_val_pr = val_pr
        best_state = {k: v.clone() for k, v in head.state_dict().items()}
    print(f"Ep {epoch:>2}  train_loss {train_loss:.4f}  val_loss {val_loss:.4f}  "
          f"val PR AUC {val_pr:.4f}  val ROC AUC {val_roc:.4f}")

head.load_state_dict(best_state)
print(f"\n✓ Best conditional-head val PR AUC (within toxic=1 subset): {best_val_pr:.4f}")


Ep  1  train_loss 0.3001  val_loss 0.2202  val PR AUC 0.5336  val ROC AUC 0.9174
Ep  2  train_loss 0.2087  val_loss 0.1966  val PR AUC 0.5626  val ROC AUC 0.9269
Ep  3  train_loss 0.1946  val_loss 0.1887  val PR AUC 0.5651  val ROC AUC 0.9286
Ep  4  train_loss 0.1892  val_loss 0.1855  val PR AUC 0.5646  val ROC AUC 0.9297
Ep  5  train_loss 0.1863  val_loss 0.1844  val PR AUC 0.5654  val ROC AUC 0.9303
Ep  6  train_loss 0.1847  val_loss 0.1843  val PR AUC 0.5656  val ROC AUC 0.9305
Ep  7  train_loss 0.1840  val_loss 0.1825  val PR AUC 0.5644  val ROC AUC 0.9308
Ep  8  train_loss 0.1831  val_loss 0.1833  val PR AUC 0.5628  val ROC AUC 0.9311
Ep  9  train_loss 0.1826  val_loss 0.1820  val PR AUC 0.5607  val ROC AUC 0.9312
Ep 10  train_loss 0.1819  val_loss 0.1827  val PR AUC 0.5582  val ROC AUC 0.9314
Ep 11  train_loss 0.1819  val_loss 0.1819  val PR AUC 0.5582  val ROC AUC 0.9314
Ep 12  train_loss 0.1815  val_loss 0.1817  val PR AUC 0.5587  val ROC AUC 0.9316
Ep 13  train_loss 0.1811  va

## Step 4 — Combined `P(severe) = P(severe | toxic=1) · P(toxic)` on validation

This is the real test. We evaluate on **all** val rows, not just the toxic-positive subset.

- `P(severe | toxic=1)` comes from our new head applied to every row's trunk features.
- `P(toxic)` comes from v7's cached val predictions in `val_inference_v7.npz`.

We compare three severe_toxic scores: v7 baseline, conditional-only (head applied to all rows, ignoring toxic prob), and combined (the hierarchical formulation).


In [13]:
# Load v7 cached val predictions (contains v7's P(toxic) we need)
_vd = np.load(CONFIG['V7_VAL_NPZ'])
v7_val_probs = _vd['val_probs']  # shape (N, 6) in labels_list order
v7_val_true  = _vd['val_true']
assert v7_val_probs.shape[0] == len(val_trunk), \
    f"Val count mismatch: v7 cache has {v7_val_probs.shape[0]}, v8 trunk has {len(val_trunk)}"

# Apply our conditional head to ALL val rows
head.eval()
with torch.no_grad():
    val_cond_logits = head(torch.tensor(val_trunk, dtype=torch.float32, device=device))
    val_cond_probs = torch.sigmoid(val_cond_logits).cpu().numpy()

# Three candidate severe_toxic scores
v7_severe = v7_val_probs[:, labels_list.index('severe_toxic')]
v7_toxic  = v7_val_probs[:, labels_list.index('toxic')]
combined_severe = val_cond_probs * v7_toxic
y_severe = val_labels_arr[:, labels_list.index('severe_toxic')]

print('Validation — severe_toxic scoring comparison:')
print('─' * 76)
print(f"{'Score':<40}{'PR AUC':>12}{'ROC AUC':>12}")
print('─' * 76)
for name, scores in [
    ('v7 baseline (independent head)', v7_severe),
    ('v8 conditional-only (ignores toxic)', val_cond_probs),
    ('v8 combined: P(sev|tox) * P(tox)',  combined_severe),
]:
    pr = average_precision_score(y_severe, scores)
    roc = roc_auc_score(y_severe, scores)
    print(f"{name:<40}{pr:>12.4f}{roc:>12.4f}")
print('─' * 76)


Validation — severe_toxic scoring comparison:
────────────────────────────────────────────────────────────────────────────
Score                                         PR AUC     ROC AUC
────────────────────────────────────────────────────────────────────────────
v7 baseline (independent head)                0.0110      0.5338
v8 conditional-only (ignores toxic)           0.5654      0.9939
v8 combined: P(sev|tox) * P(tox)              0.4124      0.9909
────────────────────────────────────────────────────────────────────────────


## Step 5 — Combined score on test set

Same combination on test. Compare test PR AUC for `severe_toxic` directly against v7's 0.344.


In [14]:
# Load v7 cached test predictions
_td = np.load(CONFIG['V7_TEST_NPZ'], allow_pickle=True)
v7_test_preds = _td['all_test_preds']      # shape (N, 6)
v7_test_labels = _td['all_test_labels']
assert v7_test_preds.shape[0] == len(test_trunk), \
    f"Test count mismatch: v7 cache {v7_test_preds.shape[0]} vs v8 trunk {len(test_trunk)}"

# Conditional head on all test rows
with torch.no_grad():
    test_cond_logits = head(torch.tensor(test_trunk, dtype=torch.float32, device=device))
    test_cond_probs = torch.sigmoid(test_cond_logits).cpu().numpy()

v7_severe_test = v7_test_preds[:, labels_list.index('severe_toxic')]
v7_toxic_test  = v7_test_preds[:, labels_list.index('toxic')]
combined_test = test_cond_probs * v7_toxic_test
y_severe_test = v7_test_labels[:, labels_list.index('severe_toxic')]

n_pos_test = int(y_severe_test.sum())
print(f"Test positives for severe_toxic: {n_pos_test} / {len(y_severe_test)} ({n_pos_test/len(y_severe_test):.2%})")
print()
print('TEST — severe_toxic scoring comparison:')
print('═' * 76)
print(f"{'Score':<40}{'PR AUC':>12}{'ROC AUC':>12}")
print('─' * 76)
results_test = {}
for name, scores in [
    ('v7 baseline (independent head)', v7_severe_test),
    ('v8 conditional-only (ignores toxic)', test_cond_probs),
    ('v8 combined: P(sev|tox) * P(tox)',  combined_test),
]:
    pr = average_precision_score(y_severe_test, scores)
    roc = roc_auc_score(y_severe_test, scores)
    results_test[name] = {'pr_auc': float(pr), 'roc_auc': float(roc)}
    print(f"{name:<40}{pr:>12.4f}{roc:>12.4f}")
print('═' * 76)

delta_pr = results_test['v8 combined: P(sev|tox) * P(tox)']['pr_auc'] - results_test['v7 baseline (independent head)']['pr_auc']
print(f"\nΔ PR AUC (combined − v7 baseline): {delta_pr:+.4f}")


Test positives for severe_toxic: 367 / 63978 (0.57%)

TEST — severe_toxic scoring comparison:
════════════════════════════════════════════════════════════════════════════
Score                                         PR AUC     ROC AUC
────────────────────────────────────────────────────────────────────────────
v7 baseline (independent head)                0.3443      0.9899
v8 conditional-only (ignores toxic)           0.3624      0.9905
v8 combined: P(sev|tox) * P(tox)              0.3615      0.9904
════════════════════════════════════════════════════════════════════════════

Δ PR AUC (combined − v7 baseline): +0.0173


In [15]:
# Save artifacts — keep them v8-tagged so v7 outputs are untouched
payload = {
    'notebook_version': 'v8-hierarchical-severe',
    'method': 'P(severe) = P(severe | toxic=1) * P(toxic_v7); conditional head = Linear(384,1) on frozen v7 trunk',
    'config': {k: v for k, v in CONFIG.items() if not callable(v)},
    'train_subset_size': int(len(X_cond_train)),
    'train_subset_severe_rate': float(y_cond_train.mean()),
    'best_val_pr_auc_within_toxic': float(best_val_pr),
    'test_severe_toxic_comparison': results_test,
    'test_delta_pr_vs_v7_baseline': float(delta_pr),
    'history': history,
}
with open('hierarchical_severe_v8.json', 'w') as f:
    json.dump(payload, f, indent=2)
print("✓ Saved hierarchical_severe_v8.json")

torch.save({'model_state_dict': head.state_dict(), 'config': CONFIG},
           'conditional_head_v8.pt')
print("✓ Saved conditional_head_v8.pt")

# Test predictions CSV with v7's 5 other labels + new combined severe
out_df = pd.DataFrame({'id': valid_test_df['id'].values})
for i, lab in enumerate(labels_list):
    if lab == 'severe_toxic':
        out_df[f'{lab}_prob_v7'] = v7_severe_test
        out_df[f'{lab}_prob_v8_combined'] = combined_test
    else:
        out_df[f'{lab}_prob_v7'] = v7_test_preds[:, i]
out_df.to_csv('test_predictions_v8_combined.csv', index=False)
print("✓ Saved test_predictions_v8_combined.csv")


✓ Saved hierarchical_severe_v8.json
✓ Saved conditional_head_v8.pt
✓ Saved test_predictions_v8_combined.csv


## Step 6 — v8.1: Unfreeze last 2 DistilBERT layers and fine-tune end-to-end

v8 (frozen trunk + logistic regression) lifted severe_toxic test PR AUC from 0.3443 → 0.3615 (+0.018). The trunk clearly has some signal the independent head missed, but the lift is modest.

**Hypothesis for v8.1:** the frozen trunk is the ceiling. If we unfreeze the last 2 DistilBERT transformer layers and let them specialize on the "severe vs toxic" distinction using only toxic-positive rows, the backbone itself can encode the finer boundary.

**Setup.**
- Fresh copy of v7 backbone + trunk, load v7 weights.
- Replace the 6-way classifier with a binary head `Linear(384, 1)` for severe_toxic.
- Freeze DistilBERT layers 0–3 (indices 0..3 of 6 total); train layers 4–5 + trunk + head.
- Train on **toxic-positive rows only** from the train split.
- Loss: BCEWithLogitsLoss. Short training (3 epochs) because the subset is small (~15K rows).

**Evaluation.** Identical to v8: val and test comparisons of v7 baseline / v8 frozen / **v8.1 unfrozen** / combined.

**Resume guard.** Checkpoint at `conditional_head_v8_1.pt` — load if present, else train.


In [16]:
# v8.1 architecture — wraps v7 backbone + trunk, replaces classifier with binary severe head.
class UnfrozenSevereHead(nn.Module):
    def __init__(self, v7_checkpoint_path, head_hidden_size=384, unfreeze_last_n=2):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.trunk = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(self.bert.config.hidden_size, head_hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        # Load v7 weights into bert + trunk (skip the 6-way classifier — we replace it)
        ckpt = torch.load(v7_checkpoint_path, map_location='cpu', weights_only=False)
        sd = ckpt['model_state_dict']
        bert_sd = {k[len('bert.'):]: v for k, v in sd.items() if k.startswith('bert.')}
        trunk_sd = {k[len('trunk.'):]: v for k, v in sd.items() if k.startswith('trunk.')}
        self.bert.load_state_dict(bert_sd, strict=True)
        self.trunk.load_state_dict(trunk_sd, strict=True)

        # Freeze all but the last N DistilBERT transformer layers
        num_layers = len(self.bert.transformer.layer)
        layers_to_freeze = num_layers - unfreeze_last_n
        for i in range(layers_to_freeze):
            for p in self.bert.transformer.layer[i].parameters():
                p.requires_grad = False
        # Always freeze embeddings (standard for fine-tuning)
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        print(f"  🔒 Frozen: embeddings + DistilBERT layers 0..{layers_to_freeze-1}")
        print(f"  🔓 Trainable: DistilBERT layers {layers_to_freeze}..{num_layers-1} + trunk + binary head")

        self.severe_head = nn.Linear(head_hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]
        h = self.trunk(pooled)
        return self.severe_head(h).squeeze(-1)


In [17]:
# Build the toxic-positive training subset as text (can't reuse trunk cache — backbone is unfrozen now)
toxic_train_idx = np.where(train_split['toxic'].values == 1)[0]
texts_train = train_split.loc[toxic_train_idx, 'comment_text'].tolist()
y_train_severe = train_split.loc[toxic_train_idx, 'severe_toxic'].values.astype(np.float32)

# Val: we'll do full-val inference for combined scoring, but for training monitoring
# use the toxic-positive val subset (within-toxic PR AUC)
toxic_val_idx = np.where(val_split['toxic'].values == 1)[0]
texts_val = val_split.loc[toxic_val_idx, 'comment_text'].tolist()
y_val_severe = val_split.loc[toxic_val_idx, 'severe_toxic'].values.astype(np.float32)

print(f"v8.1 train set: {len(texts_train):,} toxic-positive rows  "
      f"({int(y_train_severe.sum())} severe, {y_train_severe.mean():.2%})")
print(f"v8.1 val set:   {len(texts_val):,} toxic-positive rows  "
      f"({int(y_val_severe.sum())} severe)")


class TokenizedTextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        ids, mask = tokenize_text(str(self.texts[idx]))
        item = {'input_ids': torch.tensor(ids, dtype=torch.long),
                'attention_mask': torch.tensor(mask, dtype=torch.long)}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item


def collate_with_labels(batch):
    ids = pad_sequence([b['input_ids'] for b in batch], batch_first=True, padding_value=0)
    msk = pad_sequence([b['attention_mask'] for b in batch], batch_first=True, padding_value=0)
    out = {'input_ids': ids, 'attention_mask': msk}
    if 'labels' in batch[0]:
        out['labels'] = torch.stack([b['labels'] for b in batch])
    return out


train_ds_81 = TokenizedTextDataset(texts_train, y_train_severe)
val_ds_81   = TokenizedTextDataset(texts_val, y_val_severe)

train_loader_81 = DataLoader(train_ds_81, batch_size=CONFIG['BATCH_SIZE'], shuffle=True,
                              num_workers=0, collate_fn=collate_with_labels, pin_memory=True)
val_loader_81   = DataLoader(val_ds_81, batch_size=CONFIG['BATCH_SIZE'], shuffle=False,
                              num_workers=0, collate_fn=collate_with_labels, pin_memory=True)


v8.1 train set: 13,765 toxic-positive rows  (1446 severe, 10.50%)
v8.1 val set:   1,529 toxic-positive rows  (149 severe)


In [18]:
# v8.1 config
CONFIG_V81 = {
    'UNFREEZE_LAST_N': 2,
    'EPOCHS': 3,
    'LR_BACKBONE': 2e-5,     # typical DistilBERT fine-tune LR
    'LR_HEAD': 1e-3,         # new head + trunk can use higher LR
    'WEIGHT_DECAY': 1e-4,
    'CHECKPOINT': 'conditional_head_v8_1.pt',
}


In [19]:
# Train v8.1 — with resume guard. If the checkpoint exists and its config matches,
# load weights and skip training.
import os as _os
torch.manual_seed(SEED); np.random.seed(SEED)

model_81 = UnfrozenSevereHead(
    v7_checkpoint_path=CONFIG['V7_CHECKPOINT'],
    head_hidden_size=CONFIG['HEAD_HIDDEN_SIZE'],
    unfreeze_last_n=CONFIG_V81['UNFREEZE_LAST_N'],
).to(device)

_already_trained = False
if _os.path.exists(CONFIG_V81['CHECKPOINT']):
    _ck = torch.load(CONFIG_V81['CHECKPOINT'], map_location=device, weights_only=False)
    _saved_cfg = _ck.get('config_v81', {})
    _keys = ('UNFREEZE_LAST_N', 'EPOCHS')
    if all(_saved_cfg.get(k) == CONFIG_V81.get(k) for k in _keys):
        model_81.load_state_dict(_ck['model_state_dict'])
        history_81 = _ck.get('history', [])
        print(f"✓ RESUME MODE — loaded {CONFIG_V81['CHECKPOINT']}")
        print(f"  Best within-toxic val PR AUC: {_ck.get('best_val_pr_auc', 'n/a')}")
        _already_trained = True
    else:
        print(f"Config mismatch in {CONFIG_V81['CHECKPOINT']} — retraining.")

if not _already_trained:
    # Split params: backbone (low LR) vs head/trunk (higher LR)
    backbone_params = [p for n, p in model_81.bert.named_parameters() if p.requires_grad]
    head_params = (list(model_81.trunk.parameters()) + list(model_81.severe_head.parameters()))
    optimizer_81 = torch.optim.AdamW([
        {'params': backbone_params, 'lr': CONFIG_V81['LR_BACKBONE']},
        {'params': head_params,     'lr': CONFIG_V81['LR_HEAD']},
    ], weight_decay=CONFIG_V81['WEIGHT_DECAY'])

    bce_81 = nn.BCEWithLogitsLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    n_trainable = sum(p.numel() for p in model_81.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model_81.parameters())
    print(f"  Total params:     {n_total:,}")
    print(f"  Trainable params: {n_trainable:,} ({n_trainable/n_total*100:.1f}%)\n")

    history_81 = []
    best_val_pr = -1.0
    best_state = None

    for epoch in range(1, CONFIG_V81['EPOCHS'] + 1):
        model_81.train()
        total_loss = 0.0; n_seen = 0
        pbar = tqdm(train_loader_81, desc=f"v8.1 ep {epoch}/{CONFIG_V81['EPOCHS']}", leave=True)
        for batch in pbar:
            ids = batch['input_ids'].to(device, non_blocking=True)
            msk = batch['attention_mask'].to(device, non_blocking=True)
            y   = batch['labels'].to(device, non_blocking=True)
            optimizer_81.zero_grad()
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model_81(ids, msk)
                loss = bce_81(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer_81); scaler.update()
            total_loss += loss.item() * len(y); n_seen += len(y)
            pbar.set_postfix(loss=f"{total_loss/n_seen:.4f}")
        train_loss = total_loss / n_seen

        # Within-toxic val
        model_81.eval()
        val_probs_81 = []
        with torch.no_grad():
            for batch in val_loader_81:
                ids = batch['input_ids'].to(device, non_blocking=True)
                msk = batch['attention_mask'].to(device, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    lg = model_81(ids, msk)
                val_probs_81.append(torch.sigmoid(lg.float()).cpu().numpy())
        val_probs_81 = np.concatenate(val_probs_81)
        val_pr = average_precision_score(y_val_severe, val_probs_81)
        val_roc = roc_auc_score(y_val_severe, val_probs_81)
        history_81.append({'epoch': epoch, 'train_loss': train_loss,
                           'within_toxic_val_pr_auc': float(val_pr),
                           'within_toxic_val_roc_auc': float(val_roc)})
        if val_pr > best_val_pr:
            best_val_pr = val_pr
            best_state = {k: v.detach().clone().cpu() for k, v in model_81.state_dict().items()}
        print(f"  ep {epoch}: train_loss {train_loss:.4f}  within-toxic val PR AUC {val_pr:.4f}  ROC {val_roc:.4f}")

    model_81.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    torch.save({
        'model_state_dict': model_81.state_dict(),
        'config_v81': CONFIG_V81,
        'history': history_81,
        'best_val_pr_auc': float(best_val_pr),
    }, CONFIG_V81['CHECKPOINT'])
    print(f"\n✓ Saved {CONFIG_V81['CHECKPOINT']} — best within-toxic val PR AUC: {best_val_pr:.4f}")


C:\Users\berke\AppData\Local\Temp\ipykernel_23872\2657808623.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


  🔒 Frozen: embeddings + DistilBERT layers 0..3
  🔓 Trainable: DistilBERT layers 4..5 + trunk + binary head
  Total params:     66,658,561
  Trainable params: 14,471,425 (21.7%)



v8.1 ep 1/3:   0%|          | 0/216 [00:00<?, ?it/s]C:\Users\berke\AppData\Local\Temp\ipykernel_23872\2657808623.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
v8.1 ep 1/3: 100%|██████████| 216/216 [01:57<00:00,  1.84it/s, loss=0.1982]
C:\Users\berke\AppData\Local\Temp\ipykernel_23872\2657808623.py:72: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


  ep 1: train_loss 0.1982  within-toxic val PR AUC 0.5331  ROC 0.9282


v8.1 ep 2/3:   0%|          | 0/216 [00:00<?, ?it/s]C:\Users\berke\AppData\Local\Temp\ipykernel_23872\2657808623.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
v8.1 ep 2/3: 100%|██████████| 216/216 [01:58<00:00,  1.82it/s, loss=0.1841]
C:\Users\berke\AppData\Local\Temp\ipykernel_23872\2657808623.py:72: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


  ep 2: train_loss 0.1841  within-toxic val PR AUC 0.5482  ROC 0.9256


v8.1 ep 3/3:   0%|          | 0/216 [00:00<?, ?it/s]C:\Users\berke\AppData\Local\Temp\ipykernel_23872\2657808623.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
v8.1 ep 3/3: 100%|██████████| 216/216 [01:59<00:00,  1.81it/s, loss=0.1763]
C:\Users\berke\AppData\Local\Temp\ipykernel_23872\2657808623.py:72: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


  ep 3: train_loss 0.1763  within-toxic val PR AUC 0.5189  ROC 0.9223

✓ Saved conditional_head_v8_1.pt — best within-toxic val PR AUC: 0.5482


### v8.1 — full val + test evaluation

Run inference on all val + test rows (not just toxic-positive) and compute the same three-way comparison: v7 baseline / v8 frozen / v8.1 unfrozen, plus the combined variant `P(sev|tox)_v8.1 · P(tox)_v7`.


In [20]:
# Full val inference for v8.1 (all 15,958 rows)
model_81.eval()

def _infer_all(texts):
    ds = TokenizedTextDataset(list(texts))
    dl = DataLoader(ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=False,
                    num_workers=0, collate_fn=collate_varlen, pin_memory=True)
    probs = []
    with torch.no_grad():
        for batch in tqdm(dl, desc='v8.1 inference', leave=True):
            ids = batch['input_ids'].to(device, non_blocking=True)
            msk = batch['attention_mask'].to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                lg = model_81(ids, msk)
            probs.append(torch.sigmoid(lg.float()).cpu().numpy())
    return np.concatenate(probs)


# Val inference
_f_val = 'val_inference_v8_1.npz'
if _os.path.exists(_f_val):
    val_v81_probs = np.load(_f_val)['probs']
    print(f"✓ Loaded cached {_f_val}")
else:
    val_v81_probs = _infer_all(val_split['comment_text'].values)
    np.savez(_f_val, probs=val_v81_probs)
    print(f"✓ Cached {_f_val}")

y_severe_val = val_labels_arr[:, labels_list.index('severe_toxic')]
v7_severe_val = v7_val_probs[:, labels_list.index('severe_toxic')]
v7_toxic_val  = v7_val_probs[:, labels_list.index('toxic')]
combined_v81_val = val_v81_probs * v7_toxic_val

print('\nVALIDATION — severe_toxic scoring comparison (all 4 scores):')
print('─' * 82)
print(f"{'Score':<45}{'PR AUC':>12}{'ROC AUC':>12}")
print('─' * 82)
for name, scores in [
    ('v7 baseline',                                  v7_severe_val),
    ('v8 frozen conditional (Linear on trunk)',      val_cond_probs),
    ('v8.1 unfrozen (2 layers fine-tuned)',          val_v81_probs),
    ('v8.1 combined: P(sev|tox)_v81 * P(tox)_v7',    combined_v81_val),
]:
    pr = average_precision_score(y_severe_val, scores)
    roc = roc_auc_score(y_severe_val, scores)
    print(f"{name:<45}{pr:>12.4f}{roc:>12.4f}")
print('─' * 82)


v8.1 inference:   0%|          | 0/250 [00:00<?, ?it/s]C:\Users\berke\AppData\Local\Temp\ipykernel_23872\1579889646.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
v8.1 inference: 100%|██████████| 250/250 [01:35<00:00,  2.63it/s]

✓ Cached val_inference_v8_1.npz

VALIDATION — severe_toxic scoring comparison (all 4 scores):
──────────────────────────────────────────────────────────────────────────────────
Score                                              PR AUC     ROC AUC
──────────────────────────────────────────────────────────────────────────────────
v7 baseline                                        0.0110      0.5338
v8 frozen conditional (Linear on trunk)            0.5654      0.9939
v8.1 unfrozen (2 layers fine-tuned)                0.5478      0.9933
v8.1 combined: P(sev|tox)_v81 * P(tox)_v7          0.4384      0.9918
──────────────────────────────────────────────────────────────────────────────────


In [21]:
# Test inference for v8.1 (all 63,978 valid test rows)
_f_test = 'test_inference_v8_1.npz'
if _os.path.exists(_f_test):
    test_v81_probs = np.load(_f_test)['probs']
    print(f"✓ Loaded cached {_f_test}")
else:
    test_v81_probs = _infer_all(valid_test_df['comment_text'].values)
    np.savez(_f_test, probs=test_v81_probs)
    print(f"✓ Cached {_f_test}")

combined_v81_test = test_v81_probs * v7_toxic_test
y_severe_test = v7_test_labels[:, labels_list.index('severe_toxic')]

print('\nTEST — severe_toxic scoring comparison (all 4 scores):')
print('═' * 82)
print(f"{'Score':<45}{'PR AUC':>12}{'ROC AUC':>12}")
print('─' * 82)
results_v81 = {}
for name, scores in [
    ('v7 baseline',                                  v7_severe_test),
    ('v8 frozen conditional (Linear on trunk)',      test_cond_probs),
    ('v8.1 unfrozen (2 layers fine-tuned)',          test_v81_probs),
    ('v8.1 combined: P(sev|tox)_v81 * P(tox)_v7',    combined_v81_test),
]:
    pr = average_precision_score(y_severe_test, scores)
    roc = roc_auc_score(y_severe_test, scores)
    results_v81[name] = {'pr_auc': float(pr), 'roc_auc': float(roc)}
    print(f"{name:<45}{pr:>12.4f}{roc:>12.4f}")
print('═' * 82)

delta_v81_vs_v7 = results_v81['v8.1 combined: P(sev|tox)_v81 * P(tox)_v7']['pr_auc'] - results_v81['v7 baseline']['pr_auc']
delta_v81_vs_v8 = results_v81['v8.1 combined: P(sev|tox)_v81 * P(tox)_v7']['pr_auc'] - results_v81['v8 frozen conditional (Linear on trunk)']['pr_auc']
print(f"\nΔ PR AUC (v8.1 combined − v7 baseline):        {delta_v81_vs_v7:+.4f}")
print(f"Δ PR AUC (v8.1 combined − v8 frozen):          {delta_v81_vs_v8:+.4f}")


v8.1 inference:   0%|          | 0/1000 [00:00<?, ?it/s]C:\Users\berke\AppData\Local\Temp\ipykernel_23872\1579889646.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
v8.1 inference: 100%|██████████| 1000/1000 [06:16<00:00,  2.66it/s]

✓ Cached test_inference_v8_1.npz

TEST — severe_toxic scoring comparison (all 4 scores):
══════════════════════════════════════════════════════════════════════════════════
Score                                              PR AUC     ROC AUC
──────────────────────────────────────────────────────────────────────────────────
v7 baseline                                        0.3443      0.9899
v8 frozen conditional (Linear on trunk)            0.3624      0.9905
v8.1 unfrozen (2 layers fine-tuned)                0.3534      0.9891
v8.1 combined: P(sev|tox)_v81 * P(tox)_v7          0.3528      0.9896
══════════════════════════════════════════════════════════════════════════════════

Δ PR AUC (v8.1 combined − v7 baseline):        +0.0086
Δ PR AUC (v8.1 combined − v8 frozen):          -0.0096


In [22]:
# Save v8.1 artifacts
payload_v81 = {
    'notebook_version': 'v8.1-hierarchical-severe-unfrozen',
    'method': f"Unfroze last {CONFIG_V81['UNFREEZE_LAST_N']} DistilBERT layers + trunk + new binary head; "
              f"trained on toxic-positive rows only, BCE on severe_toxic.",
    'config_v81': CONFIG_V81,
    'train_subset_size': int(len(texts_train)),
    'train_subset_severe_rate': float(y_train_severe.mean()),
    'history': history_81,
    'test_comparison': results_v81,
    'test_delta_pr_vs_v7_baseline': float(delta_v81_vs_v7),
    'test_delta_pr_vs_v8_frozen':   float(delta_v81_vs_v8),
}
with open('hierarchical_severe_v8_1.json', 'w') as f:
    json.dump(payload_v81, f, indent=2)
print("✓ Saved hierarchical_severe_v8_1.json")

out_df = pd.DataFrame({'id': valid_test_df['id'].values})
out_df['severe_toxic_prob_v7']         = v7_severe_test
out_df['severe_toxic_prob_v8_frozen']  = test_cond_probs
out_df['severe_toxic_prob_v8_1']       = test_v81_probs
out_df['severe_toxic_prob_v8_1_combined'] = combined_v81_test
out_df.to_csv('test_predictions_v8_1.csv', index=False)
print("✓ Saved test_predictions_v8_1.csv")


✓ Saved hierarchical_severe_v8_1.json
✓ Saved test_predictions_v8_1.csv


## Summary

What this notebook establishes:

- **Severe-within-toxic rate** is ~10% (vs 1% overall), so a conditional head sees ~10× denser positive supervision than an independent head would.
- **Frozen-backbone proof-of-concept**: a single `Linear(384,1)` on top of v7's trunk is enough to materially move severe_toxic PR AUC — or not. If it doesn't move, the trunk doesn't encode the severe-vs-toxic distinction and we'd need to unfreeze or change the backbone.
- **Hierarchical formulation** `P(severe) = P(severe|toxic) · P(toxic)` respects the label structure: rows where v7 says "not toxic" automatically get near-zero severe probability, which is what the nested semantics dictate.

Next steps if the lift is real:
- Unfreeze the last 1–2 DistilBERT layers to let the backbone specialize.
- Apply the same pattern to `threat` if it shows nesting under any head label (it doesn't — `threat` is more independent, so hierarchical won't help there).
